In [1]:
import pyspark.sql, pyspark.sql.functions
from pyspark.sql import SparkSession

# Стартиране на SparkSession 
spark = (
    SparkSession.builder
    .appName("Cross validation example")
    .master("local[*]")
    .getOrCreate()
)
print("SparkSession стартиран успешно")
print("Spark версия:", spark.version)

SparkSession стартиран успешно
Spark версия: 4.0.1


In [6]:
# Зареждане на CSV в DataFrame
df = spark.read.csv('D:\my-repo\data\students-online.csv', header=True, inferSchema=True)
# Премахваме ID, ако не участва в модела
df = df.drop("ID")

In [14]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator

# Подготовка на характеристиките
feature_cols = ["Age","PlatformHours","OutPlatformHours"]
# Обединяване на характеристиките във вектор
assembler = VectorAssembler(inputCols=feature_cols,outputCol="features")
data = assembler.transform(df)
# Избор на колони
data = data.select("features", "Satisfaction")

# Създаване на модел
model = LinearRegression(featuresCol="features",labelCol="Satisfaction")
# Оценяване чрез RMSE
evaluator = RegressionEvaluator(labelCol="Satisfaction",predictionCol="prediction",metricName="rmse")
# Cross-validation
crossval = CrossValidator(estimator=model,
    estimatorParamMaps=[{}],   # без различни параметри
    evaluator=evaluator, numFolds=3)
# Обучение
cv_model = crossval.fit(data)

# Средна стойност на RMSE
print("Средна RMSE:")
print(cv_model.avgMetrics[0])

Средна RMSE:
0.19195358726972345
